# DICOM 3D Reconstruction — Bone and Brain Kernels

This notebook performs only the 3D reconstruction workflow:

1. locate the anonymized bone and brain CT reconstructions;
2. sort slices using DICOM patient geometry;
3. convert stored pixels to Hounsfield units;
4. isolate the principal bony structure;
5. reconstruct the bone surface with marching cubes;
6. reconstruct the brain-kernel CT as an interactive 3D volume;
7. display and export both reconstructions.

The source DICOM files are read-only and no identifying metadata is displayed.
This reconstruction is intended for visualization/research, not clinical use.

In [1]:
from pathlib import Path
import json, time

import numpy as np
import pydicom
import plotly.graph_objects as go
from IPython.display import display, HTML
from scipy import ndimage as ndi
from skimage import measure

STARTED = time.time()
DATASET_ROOT = Path(r"F:\csstdy\Single_Patient_Study")
OUTPUT_DIR = Path(r"F:\csstdy\outputs\07_3d_reconstruction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Bone surface defaults for CT. Increase the threshold for denser cortical bone.
BONE_THRESHOLD_HU = 300.0
XY_DOWNSAMPLE = 2
Z_DOWNSAMPLE = 1
KEEP_LARGEST_COMPONENT = True

assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"
print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

Dataset: F:\csstdy\Single_Patient_Study
Output: F:\csstdy\outputs\07_3d_reconstruction


## Load and geometrically sort the bone series

The dataset contains sharp bone and soft-tissue reconstructions at the same
anatomical positions. The sharp `BONE_HR60F` reconstruction is selected for
surface extraction. Slice order comes from `ImagePositionPatient` projected
onto the slice normal, not from filenames or instance numbers.

In [2]:
series_directories = sorted(path for path in DATASET_ROOT.rglob("BONE_HR60F") if path.is_dir())
if not series_directories:
    raise FileNotFoundError("No BONE_HR60F directory was found below the dataset root.")
SERIES_DIR = series_directories[0]
dicom_paths = sorted(SERIES_DIR.glob("*.dcm"))
if len(dicom_paths) < 3:
    raise RuntimeError(f"Too few DICOM slices in {SERIES_DIR}")

datasets = [pydicom.dcmread(path, force=True) for path in dicom_paths]
first = datasets[0]
orientation = np.asarray(first.ImageOrientationPatient, dtype=float)
row_direction = orientation[:3]
column_direction = orientation[3:]
slice_normal = np.cross(row_direction, column_direction)
slice_normal /= np.linalg.norm(slice_normal)

slice_locations = np.asarray([
    np.dot(np.asarray(ds.ImagePositionPatient, dtype=float), slice_normal)
    for ds in datasets
])
order = np.argsort(slice_locations)
datasets = [datasets[index] for index in order]
slice_locations = slice_locations[order]

unique_locations = np.unique(np.round(slice_locations, decimals=5))
if len(unique_locations) != len(datasets):
    raise RuntimeError("Duplicate anatomical slice locations detected in the selected series.")

slice_differences = np.diff(slice_locations)
slice_spacing = float(np.median(np.abs(slice_differences)))
row_spacing, column_spacing = (float(value) for value in first.PixelSpacing)
rows, columns = int(first.Rows), int(first.Columns)

assert all(int(ds.Rows) == rows and int(ds.Columns) == columns for ds in datasets)
assert np.allclose(slice_differences, np.median(slice_differences), atol=1e-3)

print("Selected series:", SERIES_DIR.name)
print("Slices:", len(datasets))
print("Matrix:", rows, "x", columns)
print("Voxel spacing (slice, row, column) mm:", (slice_spacing, row_spacing, column_spacing))
print("Coverage along slice normal (mm):", round(float(np.ptp(slice_locations)), 2))

Selected series: BONE_HR60F
Slices: 227
Matrix: 512 x 512
Voxel spacing (slice, row, column) mm: (0.75, 0.55574609375, 0.55574609375)
Coverage along slice normal (mm): 169.5


## Assemble the CT volume in Hounsfield units

In [3]:
volume_hu = np.empty((len(datasets), rows, columns), dtype=np.float32)
for index, ds in enumerate(datasets):
    pixels = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    volume_hu[index] = pixels * slope + intercept

finite = volume_hu[np.isfinite(volume_hu)]
print("Volume shape:", volume_hu.shape)
print("HU range:", (float(finite.min()), float(finite.max())))
print("HU percentiles [1, 50, 99]:", np.percentile(finite, [1, 50, 99]).round(1).tolist())

Volume shape: (227, 512, 512)
HU range: (-3004.0, 14722.0)


HU percentiles [1, 50, 99]: [-1050.0, -991.0, 1271.0]


## Segment the bony anatomy

The volume is downsampled only in-plane to keep the interactive mesh compact.
A mild Gaussian filter reduces staircase noise. The largest connected
high-density component is retained to suppress the scanner table and isolated
artifacts.

In [4]:
working_hu = volume_hu[::Z_DOWNSAMPLE, ::XY_DOWNSAMPLE, ::XY_DOWNSAMPLE]
smoothed_hu = ndi.gaussian_filter(working_hu, sigma=(0.45, 0.65, 0.65))
bone_mask = smoothed_hu >= BONE_THRESHOLD_HU
bone_mask = ndi.binary_closing(bone_mask, structure=ndi.generate_binary_structure(3, 1), iterations=1)

labels, component_count = ndi.label(bone_mask, structure=ndi.generate_binary_structure(3, 2))
component_sizes = np.bincount(labels.ravel())
component_sizes[0] = 0
largest_label = int(component_sizes.argmax())
largest_voxels = int(component_sizes[largest_label])

if KEEP_LARGEST_COMPONENT:
    bone_mask = labels == largest_label

if not bone_mask.any():
    raise RuntimeError("Bone segmentation is empty; lower BONE_THRESHOLD_HU.")

print("Working volume:", working_hu.shape)
print("Threshold (HU):", BONE_THRESHOLD_HU)
print("Connected high-density components:", int(component_count))
print("Retained voxels:", int(bone_mask.sum()))

Working volume: (227, 256, 256)
Threshold (HU): 300.0
Connected high-density components: 853
Retained voxels: 494281


## Reconstruct the physical 3D surface

Marching cubes returns vertices in volume coordinates. They are transformed
into the DICOM patient coordinate system using image orientation, pixel
spacing, slice normal, and the first sorted slice position.

In [5]:
mesh_spacing = (
    slice_spacing * Z_DOWNSAMPLE,
    row_spacing * XY_DOWNSAMPLE,
    column_spacing * XY_DOWNSAMPLE,
)
vertices_local, faces, normals_local, _ = measure.marching_cubes(
    bone_mask.astype(np.uint8),
    level=0.5,
    spacing=mesh_spacing,
    step_size=1,
    allow_degenerate=False,
)

origin = np.asarray(datasets[0].ImagePositionPatient, dtype=float)
vertices_patient = (
    origin
    + np.outer(vertices_local[:, 0], slice_normal)
    + np.outer(vertices_local[:, 1], column_direction)
    + np.outer(vertices_local[:, 2], row_direction)
).astype(np.float32)

normals_patient = (
    np.outer(normals_local[:, 0], slice_normal)
    + np.outer(normals_local[:, 1], column_direction)
    + np.outer(normals_local[:, 2], row_direction)
).astype(np.float32)
normal_lengths = np.linalg.norm(normals_patient, axis=1, keepdims=True)
normals_patient /= np.maximum(normal_lengths, 1e-8)

print("Mesh vertices:", f"{len(vertices_patient):,}")
print("Mesh triangles:", f"{len(faces):,}")
print("Physical bounds (mm):")
print(" min:", vertices_patient.min(axis=0).round(2).tolist())
print(" max:", vertices_patient.max(axis=0).round(2).tolist())

Mesh vertices: 264,125
Mesh triangles: 530,898
Physical bounds (mm):
 min: [-85.86000061035156, -230.44000244140625, -602.3599853515625]
 max: [49.7400016784668, -58.15999984741211, -441.1099853515625]


## Export the mesh and interactive reconstruction

In [6]:
def write_binary_ply(path, vertices, triangles, normals):
    vertices = np.asarray(vertices, dtype=np.float32)
    triangles = np.asarray(triangles, dtype=np.int32)
    normals = np.asarray(normals, dtype=np.float32)
    header = (
        "ply\nformat binary_little_endian 1.0\n"
        f"element vertex {len(vertices)}\n"
        "property float x\nproperty float y\nproperty float z\n"
        "property float nx\nproperty float ny\nproperty float nz\n"
        f"element face {len(triangles)}\n"
        "property list uchar int vertex_indices\nend_header\n"
    ).encode("ascii")
    vertex_records = np.empty(len(vertices), dtype=[
        ("x", "<f4"), ("y", "<f4"), ("z", "<f4"),
        ("nx", "<f4"), ("ny", "<f4"), ("nz", "<f4"),
    ])
    for axis, name in enumerate(("x", "y", "z")):
        vertex_records[name] = vertices[:, axis]
    for axis, name in enumerate(("nx", "ny", "nz")):
        vertex_records[name] = normals[:, axis]
    face_records = np.empty(len(triangles), dtype=[
        ("count", "u1"), ("a", "<i4"), ("b", "<i4"), ("c", "<i4")
    ])
    face_records["count"] = 3
    face_records["a"] = triangles[:, 0]
    face_records["b"] = triangles[:, 1]
    face_records["c"] = triangles[:, 2]
    with path.open("wb") as stream:
        stream.write(header)
        vertex_records.tofile(stream)
        face_records.tofile(stream)


mesh_path = OUTPUT_DIR / "CASE_001_bone_reconstruction.ply"
html_path = OUTPUT_DIR / "CASE_001_bone_reconstruction.html"
summary_path = OUTPUT_DIR / "reconstruction_summary.json"
write_binary_ply(mesh_path, vertices_patient, faces, normals_patient)

figure = go.Figure(data=[go.Mesh3d(
    x=vertices_patient[:, 0],
    y=vertices_patient[:, 1],
    z=vertices_patient[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color="#e8d6b3",
    opacity=1.0,
    flatshading=False,
    lighting=dict(ambient=0.35, diffuse=0.75, specular=0.25, roughness=0.65, fresnel=0.1),
    lightposition=dict(x=1000, y=-1000, z=1000),
    hoverinfo="skip",
)])
figure.update_layout(
    title="CASE_001 — CT Bone Surface Reconstruction",
    template="plotly_dark",
    width=900,
    height=760,
    margin=dict(l=0, r=0, t=55, b=0),
    scene=dict(
        aspectmode="data",
        xaxis_title="Patient X (mm)",
        yaxis_title="Patient Y (mm)",
        zaxis_title="Patient Z (mm)",
        xaxis=dict(showbackground=False),
        yaxis=dict(showbackground=False),
        zaxis=dict(showbackground=False),
        camera=dict(eye=dict(x=1.55, y=-1.55, z=1.15)),
    ),
)
figure.write_html(html_path, include_plotlyjs=True, full_html=True)
figure.show()

summary = {
    "source_series": SERIES_DIR.name,
    "modality": str(getattr(first, "Modality", "")),
    "source_slices": len(datasets),
    "source_shape": list(volume_hu.shape),
    "voxel_spacing_mm": [slice_spacing, row_spacing, column_spacing],
    "bone_threshold_hu": BONE_THRESHOLD_HU,
    "downsample": [Z_DOWNSAMPLE, XY_DOWNSAMPLE, XY_DOWNSAMPLE],
    "vertices": len(vertices_patient),
    "triangles": len(faces),
    "mesh_path": str(mesh_path),
    "interactive_html_path": str(html_path),
    "elapsed_seconds": time.time() - STARTED,
    "clinical_use": False,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

{
  "source_series": "BONE_HR60F",
  "modality": "CT",
  "source_slices": 227,
  "source_shape": [
    227,
    512,
    512
  ],
  "voxel_spacing_mm": [
    0.75,
    0.55574609375,
    0.55574609375
  ],
  "bone_threshold_hu": 300.0,
  "downsample": [
    1,
    2,
    2
  ],
  "vertices": 264125,
  "triangles": 530898,
  "mesh_path": "F:\\csstdy\\outputs\\07_3d_reconstruction\\CASE_001_bone_reconstruction.ply",
  "interactive_html_path": "F:\\csstdy\\outputs\\07_3d_reconstruction\\CASE_001_bone_reconstruction.html",
  "elapsed_seconds": 3.672214984893799,
  "clinical_use": false
}


## Brain-kernel 3D volume reconstruction

`BRAIN_HR36F` contains the same 227 anatomical positions reconstructed with a
soft-tissue kernel. It does not contain a binary brain mask, so this section
performs a brain-window volume rendering rather than claiming an isolated brain
segmentation. Dense bone is excluded from the displayed intensity range.

In [7]:
brain_directories = sorted(path for path in DATASET_ROOT.rglob("BRAIN_HR36F") if path.is_dir())
if not brain_directories:
    raise FileNotFoundError("No BRAIN_HR36F directory was found below the dataset root.")
BRAIN_SERIES_DIR = brain_directories[0]
brain_paths = sorted(BRAIN_SERIES_DIR.glob("*.dcm"))
brain_datasets = [pydicom.dcmread(path, force=True) for path in brain_paths]

brain_locations = np.asarray([
    np.dot(np.asarray(ds.ImagePositionPatient, dtype=float), slice_normal)
    for ds in brain_datasets
])
brain_order = np.argsort(brain_locations)
brain_datasets = [brain_datasets[index] for index in brain_order]
brain_locations = brain_locations[brain_order]

if len(brain_datasets) != len(datasets):
    raise RuntimeError("Bone and brain reconstructions have different slice counts.")
if not np.allclose(brain_locations, slice_locations, atol=1e-3):
    raise RuntimeError("Bone and brain reconstructions are not geometrically aligned.")

brain_hu = np.empty((len(brain_datasets), rows, columns), dtype=np.float32)
for index, ds in enumerate(brain_datasets):
    pixels = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    brain_hu[index] = pixels * slope + intercept

BRAIN_Z_DOWNSAMPLE = 3
BRAIN_XY_DOWNSAMPLE = 6
brain_display = brain_hu[
    ::BRAIN_Z_DOWNSAMPLE,
    ::BRAIN_XY_DOWNSAMPLE,
    ::BRAIN_XY_DOWNSAMPLE,
]

# Use the DICOM brain window. Air is set to NaN and dense bone lies outside the
# displayed range, leaving soft-tissue intensity surfaces.
brain_values = np.where(brain_display > -300.0, brain_display, np.nan).astype(np.float32)
slice_index, row_index, column_index = np.indices(brain_display.shape, dtype=np.float32)
slice_mm = slice_index.ravel() * slice_spacing * BRAIN_Z_DOWNSAMPLE
row_mm = row_index.ravel() * row_spacing * BRAIN_XY_DOWNSAMPLE
column_mm = column_index.ravel() * column_spacing * BRAIN_XY_DOWNSAMPLE
brain_origin = np.asarray(brain_datasets[0].ImagePositionPatient, dtype=np.float32)

x_patient = (
    brain_origin[0] + slice_mm * slice_normal[0]
    + row_mm * column_direction[0] + column_mm * row_direction[0]
)
y_patient = (
    brain_origin[1] + slice_mm * slice_normal[1]
    + row_mm * column_direction[1] + column_mm * row_direction[1]
)
z_patient = (
    brain_origin[2] + slice_mm * slice_normal[2]
    + row_mm * column_direction[2] + column_mm * row_direction[2]
)

brain_figure = go.Figure(data=go.Volume(
    x=x_patient,
    y=y_patient,
    z=z_patient,
    value=brain_values.ravel(),
    isomin=-5,
    isomax=80,
    opacity=0.075,
    surface_count=14,
    colorscale="Magma",
    caps=dict(x_show=False, y_show=False, z_show=False),
    hoverinfo="skip",
))
brain_figure.update_layout(
    title="CASE_001 — Brain-Kernel CT Volume Reconstruction",
    template="plotly_dark",
    width=900,
    height=760,
    margin=dict(l=0, r=0, t=55, b=0),
    scene=dict(
        aspectmode="data",
        xaxis_title="Patient X (mm)",
        yaxis_title="Patient Y (mm)",
        zaxis_title="Patient Z (mm)",
        xaxis=dict(showbackground=False),
        yaxis=dict(showbackground=False),
        zaxis=dict(showbackground=False),
        camera=dict(eye=dict(x=1.55, y=-1.55, z=1.15)),
    ),
)

brain_html_path = OUTPUT_DIR / "CASE_001_brain_kernel_volume.html"
brain_summary_path = OUTPUT_DIR / "brain_kernel_reconstruction_summary.json"
brain_figure.write_html(brain_html_path, include_plotlyjs=True, full_html=True)
brain_figure.show()

brain_summary = {
    "source_series": BRAIN_SERIES_DIR.name,
    "modality": str(getattr(brain_datasets[0], "Modality", "")),
    "source_slices": len(brain_datasets),
    "source_shape": list(brain_hu.shape),
    "voxel_spacing_mm": [slice_spacing, row_spacing, column_spacing],
    "display_downsample": [BRAIN_Z_DOWNSAMPLE, BRAIN_XY_DOWNSAMPLE, BRAIN_XY_DOWNSAMPLE],
    "display_hu_range": [-5, 80],
    "rendering": "brain-window CT volume; not a segmented brain mask",
    "interactive_html_path": str(brain_html_path),
    "clinical_use": False,
}
brain_summary_path.write_text(json.dumps(brain_summary, indent=2), encoding="utf-8")
print(json.dumps(brain_summary, indent=2))

{
  "source_series": "BRAIN_HR36F",
  "modality": "CT",
  "source_slices": 227,
  "source_shape": [
    227,
    512,
    512
  ],
  "voxel_spacing_mm": [
    0.75,
    0.55574609375,
    0.55574609375
  ],
  "display_downsample": [
    3,
    6,
    6
  ],
  "display_hu_range": [
    -5,
    80
  ],
  "rendering": "brain-window CT volume; not a segmented brain mask",
  "interactive_html_path": "F:\\csstdy\\outputs\\07_3d_reconstruction\\CASE_001_brain_kernel_volume.html",
  "clinical_use": false
}
